# Behavioral Drift Features (per customer-month)

Converts the frequency, revenue and unique-items drift features from `07_drift_features` into a per-customer-month table, so they can join onto `data/features_and_label.parquet` and feed the churn models.

**Method.** For every customer-month `t`, compare a *recent* window (months `t-2` to `t`) with the *prior* window (months `t-5` to `t-3`). Both windows are 3 months, which matches the 90-day windows in notebook 07. Only months up to `t` are used, so a row never sees information from after its own month.

**Inactive months count as zero.** The monthly table only has rows for months a customer bought something. Drift needs the quiet months too, so each customer is placed on a full monthly calendar with zeros filled in.

In [1]:
import numpy as np
import pandas as pd

WINDOW = 3  # months per window (about 90 days, as in notebook 07)

tx = pd.read_parquet('../data/outliers_handled.parquet')
tx = tx[~tx['Is_Cancellation']].copy()          # cancellations are not purchases
tx['m'] = tx['CustomerMonth'].dt.year * 12 + tx['CustomerMonth'].dt.month   # integer month index

base = pd.read_parquet('../data/features_and_label.parquet')   # the customer-month rows we attach features to
base['m'] = base['CustomerMonth'].dt.year * 12 + base['CustomerMonth'].dt.month

print('transactions:', tx.shape, '| customer-month rows:', base.shape)

transactions: (776591, 12) | customer-month rows: (20880, 10)


## 1. Drift feature builder

Wrapped in a function so the leakage test in section 3 can run it on truncated data.

In [2]:
def drift_features(tx, customers):
    """One row per (Customer ID, m) for every month on a complete calendar, for the given customers."""
    m_lo, m_hi = tx['m'].min(), tx['m'].max()
    grid = pd.MultiIndex.from_product([customers, range(m_lo, m_hi + 1)], names=['Customer ID', 'm'])

    # --- monthly invoices and revenue, zeros for months with no purchase ---
    monthly = tx.groupby(['Customer ID', 'm']).agg(freq=('Invoice', 'nunique'), rev=('Total_Price', 'sum'))
    cal = monthly.reindex(grid, fill_value=0).sort_index()

    # recent window = sum over months t-2..t, prior window = recent window 3 months earlier
    # (direct sums of shifted months: exact zeros stay zero, unlike a cumulative-sum difference)
    recent = sum(cal.groupby(level=0).shift(k, fill_value=0) for k in range(WINDOW))
    prior = recent.groupby(level=0).shift(WINDOW).fillna(0)

    # --- distinct items in each window (cannot be summed across months, so use the transactions) ---
    pairs = tx[['Customer ID', 'm', 'StockCode']].drop_duplicates()
    # an item bought in month k belongs to the recent window of months k..k+WINDOW-1
    in_window = pd.concat([pairs.assign(m=pairs['m'] + k) for k in range(WINDOW)]).drop_duplicates()
    items_recent = in_window.groupby(['Customer ID', 'm'])['StockCode'].nunique().reindex(grid, fill_value=0)
    items_prior = items_recent.groupby(level=0).shift(WINDOW).fillna(0)

    # items in the recent window that were not in the prior window
    prior_pairs = in_window.assign(m=in_window['m'] + WINDOW)      # the prior window of month t is the recent window of t-3
    chk = in_window.merge(prior_pairs, on=['Customer ID', 'm', 'StockCode'], how='left', indicator=True)
    new_items = (chk['_merge'] == 'left_only').groupby([chk['Customer ID'], chk['m']]).sum()
    new_items = new_items.reindex(grid, fill_value=0)

    f = pd.DataFrame(index=grid)
    f['freq_recent'], f['freq_prior'] = recent['freq'], prior['freq']
    f['rev_recent'], f['rev_prior'] = recent['rev'], prior['rev']
    f['items_recent'], f['items_prior'] = items_recent, items_prior
    f['items_new_recent'] = new_items

    # frequency drift
    f['freq_diff'] = f['freq_recent'] - f['freq_prior']
    f['freq_ratio'] = (f['freq_recent'] + 1) / (f['freq_prior'] + 1)
    f['freq_pct_change'] = np.where(f['freq_prior'] > 0, f['freq_diff'] / f['freq_prior'], np.nan)
    # revenue drift
    f['rev_diff'] = f['rev_recent'] - f['rev_prior']
    f['rev_log_ratio'] = np.log1p(f['rev_recent']) - np.log1p(f['rev_prior'])
    f['rev_pct_change'] = np.where(f['rev_prior'] > 0, f['rev_diff'] / f['rev_prior'], np.nan)
    # unique-items drift
    f['items_diff'] = f['items_recent'] - f['items_prior']
    f['items_ratio'] = (f['items_recent'] + 1) / (f['items_prior'] + 1)
    f['items_pct_change'] = np.where(f['items_prior'] > 0, f['items_diff'] / f['items_prior'], np.nan)
    f['items_new_share'] = np.where(f['items_recent'] > 0, f['items_new_recent'] / f['items_recent'], np.nan)

    # flags so a prior value of 0 is not mistaken for a drop in activity
    first_m = tx.groupby('Customer ID')['m'].min()
    m_idx = f.index.get_level_values('m')
    months_since_first = m_idx - first_m.reindex(f.index.get_level_values('Customer ID')).to_numpy()
    f['is_new_in_recent'] = ((months_since_first >= 0) & (months_since_first < WINDOW)).astype(int)   # first purchase falls inside the recent window
    f['has_full_history'] = (m_idx >= m_lo + 2 * WINDOW - 1).astype(int)   # prior window starts inside the dataset
    return f.reset_index()

## 2. Build and attach to the customer-month rows

In [3]:
full = drift_features(tx, base['Customer ID'].unique())
drift = base[['Customer ID', 'CustomerMonth', 'm']].merge(full, on=['Customer ID', 'm'], how='left').drop(columns='m')

assert len(drift) == len(base)
assert not drift.duplicated(['Customer ID', 'CustomerMonth']).any()
print(drift.shape)
drift.head()

(20880, 21)


,Customer ID,CustomerMonth,freq_recent,freq_prior,rev_recent,rev_prior,items_recent,items_prior,items_new_recent,freq_diff,...,freq_pct_change,rev_diff,rev_log_ratio,rev_pct_change,items_diff,items_ratio,items_pct_change,items_new_share,is_new_in_recent,has_full_history
0,12346,2010-03,1,0.0,27.05,0.00,5,0.0,5,1.0,...,NaN,27.05,3.333989,NaN,5.0,6.000000,NaN,1.0,1,0
1,12346,2010-06,1,1.0,142.31,27.05,19,5.0,19,0.0,...,0.0,115.26,1.631021,4.260998,14.0,3.333333,2.8,1.0,0,1
2,12346,2011-01,1,0.0,149.76,0.00,1,0.0,1,1.0,...,NaN,149.76,5.015689,NaN,1.0,2.000000,NaN,1.0,0,1
3,12347,2010-10,1,0.0,611.53,0.00,40,0.0,40,1.0,...,NaN,611.53,6.417598,NaN,40.0,41.000000,NaN,1.0,1,1
4,12347,2010-12,2,0.0,1323.32,0.00,70,0.0,70,2.0,...,NaN,1323.32,7.188654,NaN,70.0,71.000000,NaN,1.0,1,1


## 3. Checks

**No leakage.** Rebuild the features from transactions cut off at a past month and confirm every row up to that month is identical. If a feature used later months, these would differ.

In [4]:
T = tx['m'].quantile(0.6, interpolation='nearest')       # an arbitrary cutoff month in the middle of the data
cut = drift_features(tx[tx['m'] <= T], base['Customer ID'].unique())
a = full[full['m'] <= T].reset_index(drop=True)
b = cut.reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_exact=False, rtol=1e-9)
print('Leakage test passed: features through month index', T, 'do not change when later data is removed.')

Leakage test passed: features through month index 24135 do not change when later data is removed.


**Hand check.** One customer with gaps between purchases. Recent and prior columns should match the raw monthly numbers.

In [5]:
cid = base.groupby('Customer ID').size().sort_values().index[-1]
raw = tx[tx['Customer ID'] == cid].groupby('CustomerMonth').agg(invoices=('Invoice', 'nunique'), revenue=('Total_Price', 'sum'), items=('StockCode', 'nunique'))
print('Customer', cid, 'raw monthly activity (first 8 active months):')
display(raw.head(8))
display(drift[drift['Customer ID'] == cid][['CustomerMonth', 'freq_recent', 'freq_prior', 'freq_diff',
                                            'rev_recent', 'rev_prior', 'items_recent', 'items_prior',
                                            'is_new_in_recent', 'has_full_history']].head(8))

Customer 13199 raw monthly activity (first 8 active months):


,invoices,revenue,items
CustomerMonth,,,
2009-12,1,103.68,1
2010-01,1,382.32,2
2010-02,3,327.62,5
2010-03,3,542.42,7
2010-04,4,585.61,9
2010-05,1,390.00,6
2010-06,2,383.38,5
2010-07,2,207.36,1


,CustomerMonth,freq_recent,freq_prior,freq_diff,rev_recent,rev_prior,items_recent,items_prior,is_new_in_recent,has_full_history
3000,2009-12,1,0.0,1.0,103.68,0.00,1,0.0,1,0
3001,2010-01,2,0.0,2.0,486.00,0.00,3,0.0,1,0
3002,2010-02,5,0.0,5.0,813.62,0.00,6,0.0,1,0
3003,2010-03,7,1.0,6.0,1252.36,103.68,9,1.0,0,0
3004,2010-04,10,2.0,8.0,1455.65,486.00,13,3.0,0,0
3005,2010-05,8,5.0,3.0,1518.03,813.62,15,6.0,0,1
3006,2010-06,7,7.0,0.0,1358.99,1252.36,12,9.0,0,1
3007,2010-07,5,10.0,-5.0,980.74,1455.65,9,13.0,0,1


In [6]:
count_rev_cols = ['freq_recent', 'freq_prior', 'rev_recent', 'rev_prior', 'items_recent', 'items_prior', 'items_new_recent']
assert (drift[count_rev_cols] >= 0).all().all(), 'counts and revenue must not be negative'
assert not np.isinf(drift.select_dtypes('number')).any().any(), 'no infinite values'
pct_cols = ['freq_pct_change', 'rev_pct_change', 'items_pct_change']
print('largest absolute percent change:', drift[pct_cols].abs().max().round(1).to_dict())

largest absolute percent change: {'freq_pct_change': 12.0, 'rev_pct_change': 367.5, 'items_pct_change': 114.0}


**Missing values.** The `*_pct_change` columns and `items_new_share` are NaN by design (no prior activity, or no recent items). Everything else should have no NaNs.

In [7]:
print(drift.isna().sum()[lambda s: s > 0])
print('\nrows without a full 6-month history:', (drift['has_full_history'] == 0).sum(), 'of', len(drift))

freq_pct_change     9944
rev_pct_change      9944
items_pct_change    9944
dtype: int64

rows without a full 6-month history: 4413 of 20880


**Does it carry signal?** Churn rate by direction of revenue change. A useful drift feature should separate the groups.

In [8]:
chk = drift.merge(base[['Customer ID', 'CustomerMonth', 'is_churn']], on=['Customer ID', 'CustomerMonth'])
chk['rev_trend'] = pd.cut(chk['rev_log_ratio'], [-np.inf, -0.5, 0.5, np.inf], labels=['falling', 'flat', 'rising'])
chk.groupby('rev_trend', observed=True)['is_churn'].agg(['mean', 'size']).round(3)

,mean,size
rev_trend,,
falling,0.338,2193
flat,0.252,5391
rising,0.454,13296


## 4. Save

In [9]:
drift.to_parquet('../data/drift_features_monthly.parquet', index=False)
print('Saved data/drift_features_monthly.parquet', drift.shape)

Saved data/drift_features_monthly.parquet (20880, 21)
